# 📊 PTA Treasurer Report Generator — v4
### File-name driven · Auto-detects month · Updates correct column · GitHub integration
---
**Monthly workflow:**
1. Drop files in input folders (`quickbooks_february_2026.csv`, `givebacks_february.csv`, bank PDF)
2. Run Cells 0–10 to generate the Excel report
3. Run Cell 11 to push code changes to GitHub

**File naming:**
- QuickBooks → `quickbooks_<month>_<year>.csv`  e.g. `quickbooks_february_2026.csv`
- Givebacks  → `givebacks_<month>.csv`           e.g. `givebacks_february.csv`
- Bank PDF   → any name (month detected from content)


## Cell 0 — Install Dependencies
*Run once.*

In [1]:
import subprocess, sys, platform
print(f'Python: {sys.version}')
print(f'OS: {platform.system()}')

print('\nInstalling Python packages...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'openpyxl', 'pdfplumber', 'playwright', 'python-dotenv', '-q'])
print('  Done')

print('\nInstalling Playwright browser...')
try:
    r = subprocess.run(
        [sys.executable, '-m', 'playwright', 'install', 'chromium', '--with-deps'],
        capture_output=True, text=True)
    if r.returncode != 0:
        r2 = subprocess.run(
            [sys.executable, '-m', 'playwright', 'install', 'chromium'],
            capture_output=True, text=True)
        if r2.returncode != 0:
            print('  WARNING: Playwright install failed.')
            print('  Skip Cell 2 and upload your Givebacks CSV manually.')
        else:
            print('  Done')
    else:
        print('  Done')
except Exception as e:
    print(f'  WARNING: {e}')
    print('  Skip Cell 2 and upload your Givebacks CSV manually.')

print('\nAll dependencies ready.')


Python: 3.8.5 (default, Sep  4 2020, 02:22:02) 
[Clang 10.0.0 ]
OS: Darwin

Installing Python packages...
  Done

Installing Playwright browser...
  Done

All dependencies ready.


## Cell 1 — Configuration
*Set credentials. Month is auto-detected from filenames.*

In [23]:
from pathlib import Path
from datetime import datetime
import os, re, json, csv
from dotenv import load_dotenv

load_dotenv()

# Organisation name
ORG_NAME = os.getenv('ORG_NAME', 'Setauket School PTA')
INPUT_MONTH = "October"
FISCAL_YEAR  = "2025"   # ← the calendar year of the INPUT_MONTH
                         # July 2025 = 2025, February 2026 = 2026
#INPUT_MONTH = os.getenv('INPUT_MONTH', 'August')
#FISCAL_YEAR = os.getenv('FISCAL_YEAR', '2025')

# Single month folder for all inputs
MONTH_FOLDER = Path('input') / f'{INPUT_MONTH}_{FISCAL_YEAR}'
GB_FOLDER    = MONTH_FOLDER / 'givebacks'
QB_FOLDER    = MONTH_FOLDER
BANK_FOLDER  = MONTH_FOLDER
HISTORY_DIR = Path('data/history')

for folder in [GB_FOLDER, QB_FOLDER, BANK_FOLDER, Path('output'), HISTORY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
    
    
# Input / output folders
#GB_FOLDER   = Path('input/givebacks')/INPUT_MONTH
#QB_FOLDER   = Path('input/quickbooks')
#BANK_FOLDER = Path('input/bank')
#HISTORY_DIR = Path('data/history')

# Givebacks credentials (set here or in .env)
GIVEBACKS_EMAIL    = os.getenv('GIVEBACKS_EMAIL',    '')
GIVEBACKS_PASSWORD = os.getenv('GIVEBACKS_PASSWORD', '')
#GIVEBACKS_URL      = os.getenv('GIVEBACKS_URL',      'https://app.mygivebacks.com')
GIVEBACKS_ORG_URL = os.getenv('GIVEBACKS_ORG_URL', '')

# Prompt once if not set
if not GIVEBACKS_ORG_URL:
    GIVEBACKS_ORG_URL = input(
        'Enter your Givebacks organisation URL\n'
        '(e.g. https://yourschool.givebacks.com): '
    ).strip().rstrip('/')
    print(f'Using: {GIVEBACKS_ORG_URL}')
    print('Tip: add GIVEBACKS_ORG_URL to your .env file to skip this prompt next time.')

# GitHub settings (set here or in .env)
GITHUB_REMOTE = os.getenv('GITHUB_REMOTE', 'origin')   # remote name
GITHUB_BRANCH = os.getenv('GITHUB_BRANCH', 'main')     # branch to push to

# Fiscal year: July = index 0, June = index 11
FISCAL_MONTHS      = ['JULY','AUG','SEPT','OCT','NOV','DEC','JAN','FEB','MAR','APR','MAY','JUNE']
FISCAL_START_MONTH = 7   # July

MONTH_NAMES = {
    'january':0,'february':1,'march':2,'april':3,'may':4,'june':5,
    'july':6,'august':7,'september':8,'october':9,'november':10,'december':11,
    'jan':0,'feb':1,'mar':2,'apr':3,'jun':5,
    'jul':6,'aug':7,'sep':8,'oct':9,'nov':10,'dec':11
}

def calendar_to_fiscal(cal_month_num):
    return (cal_month_num - FISCAL_START_MONTH) % 12

def month_name_to_fiscal_index(month_name):
    cal_idx = MONTH_NAMES.get(month_name.lower())
    if cal_idx is None: return None
    return calendar_to_fiscal(cal_idx + 1)

for folder in [GB_FOLDER, QB_FOLDER, BANK_FOLDER, Path('output'), HISTORY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Organisation : {ORG_NAME}')
print(f'Fiscal year  : July (index 0) -> June (index 11)')
print(f'Givebacks    : {"credentials set" if GIVEBACKS_EMAIL else "no credentials - manual upload"}')
print(f'GitHub       : remote={GITHUB_REMOTE}  branch={GITHUB_BRANCH}')


Organisation : Setauket School PTA
Fiscal year  : July (index 0) -> June (index 11)
Givebacks    : credentials set
GitHub       : remote=origin  branch=main


In [24]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'httpx', '-q'])

0

In [25]:
#for f in GB_FOLDER.glob('*.csv'):
#    f.unlink()
#    print(f'Deleted: {f.name}')

## Cell 2 — Auto-Download Givebacks *(optional)*
Skip if uploading manually. Requires credentials in Cell 1 or `.env`.

In [43]:
import asyncio, calendar as cal_lib

async def download_givebacks(month_label, email, password, org_url, dest_folder):
    from playwright.async_api import async_playwright, TimeoutError as PWTimeout
    import csv as csv_module
    from datetime import datetime

    month_short  = month_label.split()[0].lower()
    year         = month_label.split()[1]
    target_month = datetime.strptime(month_label, '%B %Y').month
    target_year  = int(year)

    # Check if already downloaded
    existing = list(dest_folder.glob(f'givebacks_{month_short}*.csv'))
    if existing:
        print(f'Already downloaded: {len(existing)} file(s) for {month_label}')
        return existing

    print(f'Opening browser for {month_label}...')

    session_dir  = Path('data/browser_session')
    session_file = session_dir / 'givebacks_session.json'
    session_dir.mkdir(parents=True, exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage'])

        if session_file.exists():
            ctx = await browser.new_context(storage_state=str(session_file))
            print('  Loading saved session...')
        else:
            ctx = await browser.new_context()
            print('  No saved session - will login fresh')

        page = await ctx.new_page()

        try:
            # ── Check session ──────────────────────────────────────────────
            print('  -> Checking session...')
            await page.goto(f'{org_url}/payouts', timeout=60000)
            await page.wait_for_load_state('domcontentloaded')
            await page.wait_for_timeout(3000)

            current_url = page.url
            print(f'  Landing URL: {current_url}')
            login_form  = await page.locator('input[type="password"]').count()
            needs_login = ('login' in current_url.lower() or
                          'sign-in' in current_url.lower() or
                          'one-time-passcode' in current_url or
                          login_form > 0)

            if needs_login:
                print('  -> Logging in...')
                await page.goto(f'{org_url}/sign-in', timeout=30000)
                await page.wait_for_load_state('domcontentloaded')

                await page.wait_for_selector(
                    'input[type="email"], input[placeholder*="email" i]',
                    timeout=10000)
                await page.fill(
                    'input[type="email"], input[placeholder*="email" i]', email)
                await page.fill(
                    'input[type="password"], input[name="password"]', password)

                buttons = await page.locator('button').all()
                for btn in buttons:
                    txt = (await btn.inner_text()).strip()
                    if txt == 'Sign In':
                        await btn.click()
                        break
                await page.wait_for_load_state('domcontentloaded')
                await page.wait_for_timeout(2000)

                # Handle OTP
                if 'one-time-passcode' in page.url:
                    print('  OTP required - check your email')
                    code_val = input('  Enter the 6-digit code: ').strip()
                    otp_boxes = []
                    for box in await page.locator('input').all():
                        if await box.is_visible():
                            otp_boxes.append(box)
                    print(f'  Found {len(otp_boxes)} input boxes')
                    for i, digit in enumerate(code_val[:6]):
                        if i < len(otp_boxes):
                            await otp_boxes[i].click()
                            await otp_boxes[i].type(digit)
                            await page.wait_for_timeout(200)
                    await page.wait_for_timeout(1000)
                    trust = page.locator('input[type="checkbox"]')
                    if await trust.count() > 0:
                        await trust.click()
                        print('  Checked: Trust this browser for 60 days')
                    await page.wait_for_selector(
                        'button:has-text("Submit"):not([disabled])',
                        timeout=15000)
                    await page.click('button:has-text("Submit")')
                    await page.wait_for_load_state('domcontentloaded')
                    await page.wait_for_timeout(2000)
                    print('  OTP verified')

                print(f'  URL after login: {page.url}')
                if 'one-time-passcode' in page.url or 'login' in page.url.lower():
                    raise Exception('Login failed - OTP may be incorrect or expired')
                print('  Logged in successfully')

                # Save session
                await ctx.storage_state(path=str(session_file))
                print(f'  Session saved -> {session_file}')

                # Navigate to payouts
                await page.goto(f'{org_url}/payouts', timeout=60000)
                await page.wait_for_load_state('domcontentloaded')
                await page.wait_for_timeout(2000)

            else:
                print('  Session valid - skipping login')

            # ── Capture auth token ─────────────────────────────────────────
            auth_token = None

            async def handle_response(response):
                nonlocal auth_token
                if 'api.givebacks.com' in response.url:
                    try:
                        req_headers = response.request.headers
                        if 'authorization' in req_headers:
                            auth_token = req_headers['authorization']
                    except:
                        pass

            page.on('response', handle_response)

            # Reload payouts to trigger API calls
            print('  -> Checking session...')
            await page.goto(f'{org_url}/payouts', timeout=60000)
            await page.wait_for_load_state('domcontentloaded')
            await page.wait_for_timeout(3000)
            
            # Try to get auth token from localStorage
            auth_token = await page.evaluate('''() => {
                // Try common storage keys
                const keys = ['token', 'authToken', 'auth_token', 
                              'access_token', 'jwt', 'Bearer'];
                for (const key of keys) {
                    const val = localStorage.getItem(key);
                    if (val) return val;
                }
                // Try all localStorage keys
                for (let i = 0; i < localStorage.length; i++) {
                    const key = localStorage.key(i);
                    const val = localStorage.getItem(key);
                    if (val && val.startsWith('ey')) return val;  // JWT token
                }
                return null;
            }''')
            
            print(f'  Auth token from storage: {auth_token[:50] if auth_token else "NOT FOUND"}')
            
            # Also try cookies
            cookies = await ctx.cookies()
            print('  Cookies found:')
            for cookie in cookies:
                if 'token' in cookie['name'].lower() or 'auth' in cookie['name'].lower():
                    print(f'    {cookie["name"]}: {cookie["value"][:50]}')
            
            
            # Dismiss popup
            try:
                await page.locator('[aria-label="Close"], button:has-text("×")').first.click(timeout=2000)
                print('  Dismissed popup')
            except:
                await page.keyboard.press('Escape')
            await page.wait_for_timeout(500)

            print(f'  Auth token: {auth_token[:50] if auth_token else "NOT FOUND"}')

            # ── Find payouts for target month ──────────────────────────────
            # ── Call Givebacks API directly with cookies ───────────────────
            # ── Call Givebacks API directly ────────────────────────────────
            import httpx, calendar as cal

            all_cookies = await ctx.cookies()
            cookie_dict = {c['name']: c['value'] for c in all_cookies
                          if 'givebacks.com' in c.get('domain', '')}

            cause_id   = '96011dc5-4a0d-4502-a2a2-909b951d7579'
            last_day   = cal.monthrange(target_year, target_month)[1]
            start_date = f'{target_year}-{target_month:02d}-01'
            end_date   = f'{target_year}-{target_month:02d}-{last_day:02d}'

            # Fetch all payouts with pagination
            all_payouts = []
            offset = 0
            while True:
                async with httpx.AsyncClient() as client:
                    r = await client.get(
                        f'https://api.givebacks.com/services/payout/payouts'
                        f'?cause_id={cause_id}&limit=25&offset={offset}',
                        cookies=cookie_dict,
                        headers={
                            'Origin':  org_url,
                            'Referer': f'{org_url}/payouts',
                        },
                        timeout=30
                    )
                if r.status_code != 200:
                    print(f'  API error: {r.status_code}')
                    break
                data = r.json()
                payouts = data.get('payouts', [])
                all_payouts.extend(payouts)
                if not data.get('meta', {}).get('has_more', False):
                    break
                offset += 25

            print(f'  Total payouts fetched: {len(all_payouts)}')

            # Filter payouts - use arrival_date within month OR first week of next month
            import calendar as cal
            last_day   = cal.monthrange(target_year, target_month)[1]
            
            # Next month for overflow payouts
            if target_month == 12:
                next_month = 1
                next_year  = target_year + 1
            else:
                next_month = target_month + 1
                next_year  = target_year
            
            payout_urls = []
            for payout in all_payouts:
                arrival = payout.get('arrival_date', '')[:10]  # "2025-08-01"
                amount  = payout.get('amount', 0)
                
                # Match payouts arriving in target month OR first 7 days of next month
                in_target_month = arrival.startswith(f'{target_year}-{target_month:02d}-')
                in_next_month   = (arrival.startswith(f'{next_year}-{next_month:02d}-') and
                                   int(arrival[8:10]) <= 7)
                
                if (in_target_month or in_next_month) and payout.get('status') == 'paid':
                    pid = payout.get('uuid')
                    if pid:
                        payout_urls.append(
                            (pid, f'{org_url}/payouts/{pid}/summary'))
                        print(f'  Matched: {pid}  '
                              f'${amount/100:.2f}  '
                              f'arrival={arrival}')

            seen = set()
            payout_urls = [x for x in payout_urls
                          if x[0] not in seen and not seen.add(x[0])]
            print(f'  Found {len(payout_urls)} payout(s) for {month_label}')
        
            if not payout_urls:
                print(f'  WARNING: No payouts found for {month_label}')
                return []

            # ── Scrape each payout summary ─────────────────────────────────
            saved_files = []
            for i, (payout_id, summary_url) in enumerate(payout_urls, 1):
                print(f'  -> Scraping payout {i}/{len(payout_urls)}: {payout_id}')
                await page.goto(summary_url, timeout=60000)
                await page.wait_for_load_state('domcontentloaded')
                await page.wait_for_timeout(2000)

                rows_data = []
                table_rows = await page.locator('table tbody tr').all()
                for tr in table_rows:
                    cells = await tr.locator('td').all()
                    if len(cells) < 3:
                        continue
                    item = (await cells[0].inner_text()).strip()
                    if not item or item in ('Item', 'Total'):
                        continue
                    category = (await cells[1].inner_text()).strip()
                    try:    txns = int((await cells[2].inner_text()).strip())
                    except: txns = 0
                    try:
                        total = float(
                            (await cells[3].inner_text()).strip()
                            .replace('$','').replace(',',''))
                    except: total = 0.0
                    rows_data.append({
                        'Item':                item,
                        'Categories':          category,
                        'No. of Transactions': txns,
                        'Total':               f'${total:,.2f}',
                    })

                if not rows_data:
                    print(f'    WARNING: No data scraped from {summary_url}')
                    continue

                dest_folder.mkdir(parents=True, exist_ok=True)
                fname = dest_folder / f'givebacks_{month_short}_{payout_id}.csv'
                with open(fname, 'w', newline='') as f:
                    writer = csv_module.DictWriter(
                        f, fieldnames=['Item','Categories',
                                       'No. of Transactions','Total'])
                    writer.writeheader()
                    writer.writerows(rows_data)
                print(f'    Saved: {fname.name}  ({len(rows_data)} items)')
                saved_files.append(fname)

            print(f'\n  Done: {len(saved_files)} CSV file(s) saved to {dest_folder}')
            return saved_files

        except Exception as e:
            await page.screenshot(path='debug_error.png')
            raise Exception(f'{e}\nCheck debug_error.png for page state')
        finally:
            await ctx.close()
            await browser.close()


# ── Run ───────────────────────────────────────────────────────────────────────
year_str = FISCAL_YEAR
label    = f'{INPUT_MONTH.capitalize()} {year_str}'
print(f'Downloading Givebacks for: {label}  (from INPUT_MONTH in Cell 1)')

if not GIVEBACKS_EMAIL or not GIVEBACKS_PASSWORD:
    print('No credentials found.')
    print('Set GIVEBACKS_EMAIL and GIVEBACKS_PASSWORD in your .env file')
    print('OR place CSV files manually in:', GB_FOLDER)
else:
    try:
        await download_givebacks(
            label, GIVEBACKS_EMAIL, GIVEBACKS_PASSWORD,
            GIVEBACKS_ORG_URL, GB_FOLDER)
    except Exception as e:
        print(f'Download failed: {e}')
        
        print('Upload Givebacks CSV manually to', GB_FOLDER)

Already downloaded: 7 file(s) for October 2025


## Cell 3 — Get Consistent Files across all three Platforms for a given month
Reads filenames and PDF content to determine which fiscal month to update.


In [110]:
import pdfplumber
def detect_month_from_filename(filepath):
    name  = filepath.stem.lower()
    parts = re.split(r'[_\-\s]+', name)
    month_str = None; year_str = None; fiscal_idx = None
    for part in parts:
        if part in MONTH_NAMES and month_str is None:
            month_str  = part
            fiscal_idx = month_name_to_fiscal_index(part)
        if re.match(r'^20\d{2}$', part) and year_str is None:
            year_str = part
    if month_str is None: return None, None
    year_str = year_str or FISCAL_YEAR
    month_label = f'{month_str.capitalize()} {year_str}'
    return month_label, fiscal_idx

def detect_month_from_pdf(pdf_path):
    # Try filename first
    lbl, idx = detect_month_from_filename(pdf_path)
    if lbl:
        print(f'  Bank month from filename: {lbl}')
        return lbl, idx

    # Fall back to PDF content
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text() or ''
        m = re.search(
        r'(January|February|March|April|May|June|July|August|'
        r'September|October|November|December)\s+(\d{1,2}),\s+(\d{4})'
        r'\s*through', text, re.IGNORECASE)
        if m:
            return f'{m.group(1)} {m.group(3)}', month_name_to_fiscal_index(m.group(1))
    except Exception as e:
        print(f'  Could not read PDF content: {e}')
    return None, None

# QuickBooks
qb_files = sorted(f for f in QB_FOLDER.glob('*.csv') 
                  if INPUT_MONTH.lower() in f.name.lower())
if not qb_files:
    all_qb = sorted(QB_FOLDER.glob('*.csv'))
    raise FileNotFoundError(
        f'No QuickBooks file found for {INPUT_MONTH}\n'
        f'Files available: {[f.name for f in all_qb]}\n'
        f'Expected: quickbooks_{INPUT_MONTH.lower()}_2026.csv'
    )
QB_FILE = qb_files[-1]

QB_MONTH_LABEL, QB_FISCAL_IDX = detect_month_from_filename(QB_FILE)

print(f'QuickBooks : {QB_FILE.name}  ->  {QB_MONTH_LABEL}  (fiscal index {QB_FISCAL_IDX}: {FISCAL_MONTHS[QB_FISCAL_IDX]})')
if QB_MONTH_LABEL.split()[0].lower() != INPUT_MONTH.lower():
    raise ValueError(
        f'\nMONTH MISMATCH - QuickBooks file!\n'
        f'  INPUT_MONTH set to : {INPUT_MONTH}\n'
        f'  QB file detected   : {QB_MONTH_LABEL}\n'
        f'  Fix: rename file to quickbooks_{INPUT_MONTH.lower()}_2026.csv\n'
        f'       or update INPUT_MONTH in Cell 1'
    )
print(f'  Month matches INPUT_MONTH ✅')

# Givebacks
#gb_files = sorted(GB_FOLDER.glob('givebacks_*.csv'))
gb_files = sorted(f for f in GB_FOLDER.rglob('*.csv') if 'givebacks' in f.name.lower())
if not gb_files:
    raise FileNotFoundError(
        f'No Givebacks files in {GB_FOLDER}\nName them: givebacks_february.csv')
GB_FILE_INFO = []
print(f'\nGivebacks :')
for f in gb_files:
    lbl, idx = detect_month_from_filename(f)
    if lbl:
        print(f'  {f.name}  ->  {lbl}  (fiscal index {idx})')
        GB_FILE_INFO.append((f, lbl, idx))
    else:
        print(f'  {f.name}  ->  WARNING: could not detect month, skipping')

# Validate all Givebacks files match INPUT_MONTH
for f, lbl, idx in GB_FILE_INFO:
    if lbl.split()[0].lower() != INPUT_MONTH.lower():
        raise ValueError(
            f'\nMONTH MISMATCH - Givebacks file!\n'
            f'  INPUT_MONTH set to  : {INPUT_MONTH}\n'
            f'  File {f.name} detected as: {lbl}\n'
            f'  Fix: move file to correct month folder\n'
            f'       or update INPUT_MONTH in Cell 1'
        )
print(f'  All Givebacks files match INPUT_MONTH ✅')
        
pdf_files = sorted(BANK_FOLDER.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDF in {BANK_FOLDER}')

# Try to find file matching INPUT_MONTH first
matching = [f for f in pdf_files if INPUT_MONTH.lower() in f.name.lower()]
BANK_FILE = matching[-1] if matching else pdf_files[-1]
if not matching:
    print(f'  WARNING: No bank PDF found matching {INPUT_MONTH} - using {BANK_FILE.name}')
    
BANK_FILE = pdf_files[-1]
BANK_MONTH_LABEL, BANK_FISCAL_IDX = detect_month_from_pdf(BANK_FILE)
print(f'\nBank PDF  : {BANK_FILE.name}  ->  ', end='')
if BANK_MONTH_LABEL:
    print(f'{BANK_MONTH_LABEL}  (fiscal index {BANK_FISCAL_IDX})')
else:
    BANK_MONTH_LABEL, BANK_FISCAL_IDX = QB_MONTH_LABEL, QB_FISCAL_IDX
    print(f'month not detected - using QB month: {QB_MONTH_LABEL}')

if BANK_MONTH_LABEL and BANK_MONTH_LABEL.split()[0].lower() != INPUT_MONTH.lower():
    raise ValueError(
        f'\nMONTH MISMATCH - Bank PDF!\n'
        f'  INPUT_MONTH set to : {INPUT_MONTH}\n'
        f'  Bank PDF detected  : {BANK_MONTH_LABEL}\n'
        f'  Fix: check you uploaded the correct month statement'
    )
if BANK_MONTH_LABEL:
    print(f'  Bank month matches INPUT_MONTH ✅')
    
MONTH_LABEL = QB_MONTH_LABEL
FISCAL_IDX  = QB_FISCAL_IDX
safe_month  = MONTH_LABEL.replace(' ', '_')
OUTPUT_FILE = Path(f'output/Treasurer_Report_{safe_month}.xlsx')

print(f" {'='*55}")
print(f'  Report month : {MONTH_LABEL}')
print(f'  Fiscal index : {FISCAL_IDX}  -> column {FISCAL_MONTHS[FISCAL_IDX]} in budget sheets')
print(f'  Output       : {OUTPUT_FILE}')
print(f"{'='*55}")


QuickBooks : quickbooks_october_2025.csv  ->  October 2025  (fiscal index 3: OCT)
  Month matches INPUT_MONTH ✅

Givebacks :
  givebacks_october_po_1SDwy04TnYF4pDk8QbC52BJe.csv  ->  October 2025  (fiscal index 3)
  givebacks_october_po_1SGU5K4TnYF4pDk8ZVpTiwWq.csv  ->  October 2025  (fiscal index 3)
  givebacks_october_po_1SHxfy4TnYF4pDk8B57GMtie.csv  ->  October 2025  (fiscal index 3)
  givebacks_october_po_1SJ1GX4TnYF4pDk8dGPOLPNu.csv  ->  October 2025  (fiscal index 3)
  givebacks_october_po_1SLZbN4TnYF4pDk8CyNZv1u8.csv  ->  October 2025  (fiscal index 3)
  givebacks_october_po_1SO5s84TnYF4pDk8T4z0Vplz.csv  ->  October 2025  (fiscal index 3)
  givebacks_october_po_1SQdlL4TnYF4pDk8eh5xCzKH.csv  ->  October 2025  (fiscal index 3)
  All Givebacks files match INPUT_MONTH ✅
  Bank month from filename: October 2025

Bank PDF  : Chase_October_2025.pdf  ->  October 2025  (fiscal index 3)
  Bank month matches INPUT_MONTH ✅
  Report month : October 2025
  Fiscal index : 3  -> column OCT in bu

In [111]:
print('QB_FILE exists:  ', QB_FILE.exists())
print('GB files found:  ', list(GB_FOLDER.glob('*.csv')))
print('Bank files found:', list(BANK_FOLDER.glob('*.pdf')))

QB_FILE exists:   True
GB files found:   [PosixPath('input/October_2025/givebacks/givebacks_october_po_1SLZbN4TnYF4pDk8CyNZv1u8.csv'), PosixPath('input/October_2025/givebacks/givebacks_october_po_1SHxfy4TnYF4pDk8B57GMtie.csv'), PosixPath('input/October_2025/givebacks/givebacks_october_po_1SDwy04TnYF4pDk8QbC52BJe.csv'), PosixPath('input/October_2025/givebacks/givebacks_october_po_1SGU5K4TnYF4pDk8ZVpTiwWq.csv'), PosixPath('input/October_2025/givebacks/givebacks_october_po_1SJ1GX4TnYF4pDk8dGPOLPNu.csv'), PosixPath('input/October_2025/givebacks/givebacks_october_po_1SQdlL4TnYF4pDk8eh5xCzKH.csv'), PosixPath('input/October_2025/givebacks/givebacks_october_po_1SO5s84TnYF4pDk8T4z0Vplz.csv')]
Bank files found: [PosixPath('input/October_2025/Chase_October_2025.pdf')]


## Cell 4 — Parse Files & Save to History

In [112]:
pdf_files = sorted(BANK_FOLDER.glob('*.pdf'))
print (pdf_files)

[PosixPath('input/October_2025/Chase_October_2025.pdf')]


In [113]:
# ── Load parsers and builders from .py files ──────────────────────────────
import importlib, sys

# Force remove cached versions
for mod in ['parsers', 'builders']:
    if mod in sys.modules:
        del sys.modules[mod]

# Fresh import
import parsers, builders
from parsers import parse_quickbooks_detail, parse_givebacks_files, parse_chase_pdf
from builders import (build_treasurer, build_budget, build_givebacks,
                      build_manifest, build_ytd_summary)
print('✅ Parsers and builders loaded fresh')

try:
    qb = parse_quickbooks_detail(QB_FOLDER)
    print('QB OK:', qb['income_total'])
except Exception as e:
    print('QB FAILED:', e)

# Step 2 - test Givebacks alone
try:
    givebacks = parse_givebacks_files(GB_FILE_INFO)
    print('Givebacks OK:', len(givebacks))
except Exception as e:
    print('Givebacks FAILED:', e)

# Step 3 - test Bank alone
try:
    bank = parse_chase_pdf(BANK_FILE)
    print(f'Bank OK: ${bank["beginning_balance"]:,.2f}')
except Exception as e:
    print(f'Bank FAILED: {e}')
    
HISTORY_DIR.mkdir(parents=True, exist_ok=True)
safe_month = MONTH_LABEL.replace(' ', '_')

# Save to history

try:
    safe_month = MONTH_LABEL.replace(' ', '_')
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    hist_entry = {
        'month_label':     MONTH_LABEL,
        'fiscal_index':    FISCAL_IDX,
        'income':          qb['income'],
        'expenses':        qb['expenses'],
        'income_total':    qb['income_total'],
        'expense_total':   qb['expense_total'],
        'net_income':      qb['net_income'],
        'givebacks_total': sum(g['total'] for g in givebacks),
        'generated_at':    datetime.today().isoformat(),
    }
    # Always overwrite existing history for current month
    hist_path = HISTORY_DIR / f'{safe_month}.json'
    if hist_path.exists():
        print(f'  Overwriting existing history for {MONTH_LABEL}')
    hist_path.write_text(json.dumps(hist_entry, indent=2))
    print(f'History saved -> {hist_path}')
except Exception as e:
    print(f'\nWARNING: Could not save history: {e}')
    print('Report generation will continue.')


✅ Parsers and builders loaded fresh
  QuickBooks: quickbooks_october_2025.csv
  Encoding  : utf-8-sig
  Period    : October 1-31, 2025
  Income    : $2,683.52  (10 categories)
  Expenses  : $17,851.64  (13 categories)
  Transactions: 43
QB OK: 2683.52
  Givebacks : 9 items  total=$1,250.00
Givebacks OK: 9
  Bank      : Chase_October_2025.pdf
  Period    : October 01, 2025throughOctober 31, 2025
  Beginning : $49,954.22  Ending: $34,446.10
  Deposits      : 6  total=$2,243.52
  Checks        : 13  total=$17,294.64
  Withdrawals   : $457.00
  Fees          : $0.00
Bank OK: $49,954.22
  Overwriting existing history for October 2025
History saved -> data/history/October_2025.json


In [114]:
import json
from pathlib import Path

hist_files = sorted(Path('data/history').glob('*.json'))
print(f'History files found: {len(hist_files)}')
for hf in hist_files:
    e = json.loads(hf.read_text())
    print(f'  {hf.name}')
    print(f'    month_label  : {e["month_label"]}')
    print(f'    fiscal_index : {e["fiscal_index"]}')
    print(f'    income_total : ${e["income_total"]:,.2f}')
    print(f'    expense_total: ${e["expense_total"]:,.2f}')

History files found: 6
  April_2026.json
    month_label  : April 2026
    fiscal_index : 9
    income_total : $31,732.00
    expense_total: $21,534.79
  August_2025.json
    month_label  : August 2025
    fiscal_index : 1
    income_total : $525.14
    expense_total: $391.37
  July_2025.json
    month_label  : July 2025
    fiscal_index : 0
    income_total : $0.00
    expense_total: $1,439.34
  March_2026.json
    month_label  : March 2026
    fiscal_index : 8
    income_total : $8,405.21
    expense_total: $2,552.28
  October_2025.json
    month_label  : October 2025
    fiscal_index : 3
    income_total : $2,683.52
    expense_total: $17,851.64
  September_2025.json
    month_label  : September 2025
    fiscal_index : 2
    income_total : $20,661.97
    expense_total: $2,032.28


In [115]:
print(f'total_deposits    : ${bank["total_deposits"]:,.2f}')
print(f'total_checks      : ${bank["total_checks"]:,.2f}')
print(f'total_withdrawals : ${bank["total_withdrawals"]:,.2f}')
print(f'total_fees        : ${bank["total_fees"]:,.2f}')
print(f'beginning_balance : ${bank["beginning_balance"]:,.2f}')
print(f'ending_balance    : ${bank["ending_balance"]:,.2f}')
print(f'\nWithdrawal items found: {len(bank["withdrawals"])}')
for w in bank["withdrawals"]:
    print(f'  {w["date"]}  {w["description"][:45]:<46} ${w["amount"]:>10,.2f}')

calc = (bank['beginning_balance']
        + bank['total_deposits']
        - bank['total_checks']
        - bank.get('total_withdrawals', 0.0)
        - bank['total_fees'])
print(f'\nCalculated ending : ${calc:,.2f}')
print(f'Actual ending     : ${bank["ending_balance"]:,.2f}')
print(f'Difference        : ${calc - bank["ending_balance"]:,.2f}')

total_deposits    : $2,243.52
total_checks      : $17,294.64
total_withdrawals : $457.00
total_fees        : $0.00
beginning_balance : $49,954.22
ending_balance    : $34,446.10

Withdrawal items found: 0

Calculated ending : $34,446.10
Actual ending     : $34,446.10
Difference        : $0.00


## Cell 5 — Build Actuals from History
Assembles the 12-column actuals arrays from all stored monthly JSON files.

In [116]:
from collections import defaultdict
def load_all_actuals():
    hist_files = sorted(HISTORY_DIR.glob('*.json'))
    if not hist_files:
        print('No history yet - actuals will be zeros until months are processed.')
        return {}, {}
    income_actuals  = defaultdict(lambda: [0.0]*12)
    expense_actuals = defaultdict(lambda: [0.0]*12)
    for hf in hist_files:
        try:
            entry = json.loads(hf.read_text())
            idx   = entry.get('fiscal_index')
            if idx is None: continue
            for item, val in entry.get('income', {}).items():
                income_actuals[item][idx] = val
            for item, val in entry.get('expenses', {}).items():
                expense_actuals[item][idx] = val
            print(f'  Loaded: {hf.stem:<25} -> {entry["month_label"]:<18} (index {idx}: {FISCAL_MONTHS[idx]})')
        except Exception as e:
            print(f'  Warning: could not load {hf.name}: {e}')
    return dict(income_actuals), dict(expense_actuals)

print('Loading actuals from history...')
INCOME_ACTUALS_LIVE, EXPENSE_ACTUALS_LIVE = load_all_actuals()
print(f'\nIncome items tracked : {len(INCOME_ACTUALS_LIVE)}')
print(f'Expense items tracked: {len(EXPENSE_ACTUALS_LIVE)}')
print(f'\nValues for {MONTH_LABEL} (column {FISCAL_MONTHS[FISCAL_IDX]}):')
income_this_month = {k:v[FISCAL_IDX] for k,v in INCOME_ACTUALS_LIVE.items() if v[FISCAL_IDX]}
expense_this_month = {k:v[FISCAL_IDX] for k,v in EXPENSE_ACTUALS_LIVE.items() if v[FISCAL_IDX]}
print('  Income:')
for k,v in income_this_month.items():  print(f'    {k:<40} ${v:>10,.2f}')
print('  Expenses:')
for k,v in expense_this_month.items(): print(f'    {k:<40} ${v:>10,.2f}')


Loading actuals from history...
  Loaded: April_2026                -> April 2026         (index 9: APR)
  Loaded: August_2025               -> August 2025        (index 1: AUG)
  Loaded: July_2025                 -> July 2025          (index 0: JULY)
  Loaded: March_2026                -> March 2026         (index 8: MAR)
  Loaded: October_2025              -> October 2025       (index 3: OCT)
  Loaded: September_2025            -> September 2025     (index 2: SEPT)

Income items tracked : 21
Expense items tracked: 28

Values for October 2025 (column OCT):
  Income:
    Birthday Books-Income                    $    390.00
    Fast Athletics                           $    910.00
    Family Membership                        $     50.00
    Standard                                 $    285.00
    Teachers                                 $    240.00
    Graduating Class Dues                    $    300.00
    Membership                               $     45.00
    Student                

## Cell 6 — Annual Budget Data
*Edit once per fiscal year (July). Last Year = prior year totals.*

In [117]:
# Maps QuickBooks category names -> Budget category names
QB_TO_BUDGET_MAP = {
    # Income
    'Fast Athletics':          'FAST',
    'Basket Dinner':           'Ticket & Raffle Sales',
    'Contribution':            'Ticket & Raffle Sales',
    'Skate Night':             'The Night at Rinx',
    'Birthday Books-Income':   'Birthday Books',
    'Membership - Teachers':   'Teachers',
    'Lawn Signs':              'Lawn Signs',
    'Plant Sale':              'Plant Sale',
    'Staff Appreciation':      'Staff Appreciation',
    'Talent Show Income':      'Talent Show',
    'Fall Pictures':           'Fall Pictures',
    'shop to give donations': 'Donations',
    'Family Membership':      'Family',
    'Standard':               'Single',
    'Teachers':               'Teachers',
    'Membership':            'Single',
    'Misc MemberHub Income': 'Donations',
    'Birthday Books-Income': 'Birthday Books',
    'Fast Athletics':        'FAST',
    'Trunk or Treat':        'Treat or Trunk',
    'Graduating Class Dues': 'Family Contributions',
    # Expenses
    'Accounting Expense (Quickbooks)': 'Accounting Quickbooks',
    'Bank Charges & Fees':             'Bank Services',
    'Arts In Education':               'Cultural Arts',
    'Scholastic Book Fairs':           'Book Fair',
    'Movie Night':                     'Outdoor Movie',
    'Basket dinner':                   'Entertainment',
    'Basket Dinner - Venue':           'Venue',
    'Welcome Back Staff':              'Welcome Back Breakfast',
    'Gingerbread U':                   'Gingerbread U',
    'Multicultural Night':             'Multicultural Night',
    'Science Fair':                    'Science Fair',
    'Talent Show':                     'Talent Show',
    'Fast Athletics':                  'FAST',
    'Birthday Books-Income':           'Birthday Books',
    'Membership - Teachers':           'Teachers',
    'MEMBERSHIP EXPENSE':              'Membership Expenses',
    'Skate Night':                     'The Night at Rinx',
    '6th Grade Events':                'Grad Class Events',
    'Website':                         'Website & Remind App',
}

# Format: 'Item': (last_year_actual, annual_budget)
INCOME_BUDGET = {
    'Fundraising': {
        'Birthday Books':  (2310.00, 2000.00), 'Book Fair':       (9118.36,  500.00),
        'Croc Charms':     (405.00,   100.00), 'Fall Pictures':   (3350.25, 3000.00),
        'FAST':            (26370.00,1000.00), 'Holiday Boutique':(13417.00,7500.00),
        'Plant Sale':      (8202.68, 8000.00), 'Spiritwear':      (1253.96, 1500.00),
        'Spring Pictures': (0.00,       0.00),
        'Ticket & Raffle Sales': (22165.00, 15000.00),
        'Sponsors':              (7450.00,   4000.00),
    },
    'Membership': {
        'Single':(2580.00,2000.00), 'Family':(1675.00,1000.00),
        'Donations':(290.63,100.00), 'Teachers':(0.00,165.00), 'Student':(0.00,5.00),
    },
    'Program': {
        'Gingerbread U':(3515.00,0.00), 'Staff Appreciation':(1625.00,1000.00),
        'Talent Show':(1070.00,750.00),
    },
    'Grad Class Activities': {
        'Family Contributions':(6480.00,3900.00), 'Lawn Signs':(2560.00,750.00),
        'Treat or Trunk':(1250.00,975.00), 'The Night at Rinx':(0.00,0.00),
    },
}

EXPENSE_BUDGET = {
    'Admin/General/Membership': {
        'Accountant':(650.00,650.00), 'Bank Services':(234.94,200.00),
        'Insurance':(350.00,350.00), 'Supplies':(450.22,500.00),
        'Accounting Quickbooks':(410.66,1300.00), 'Training':(68.90,100.00),
        'Website & Remind App':(344.01,1000.00), 'Event Equipment':(563.32,1000.00),
        'Council Dues':(125.00,150.00), 'Membership Expenses':(1034.00,1500.00),
    },
    'Fundraising': {
        'Entertainment':   (1405.42, 1000.00),
        'Raffles':         (1831.70, 2000.00),
        'Venue':           (8265.20, 10000.00),
        'Birthday Books':(503.39,500.00), 'Book Fair':(9523.91,10000.00),
        'Croc Charms':(205.00,0.00), 'Fall Pictures':(0.00,0.00),
        'FAST':(24000.00,17000.00), 'Holiday Boutique':(11714.86,10000.00),
        'Plant Sale':(5615.95,6000.00), 'Spiritwear':(0.00,1000.00),
        'Spring Pictures':(0.00,0.00),
    },
    'Programs': {
        'Bus Driver Appreciation':(240.00,300.00), 'Electric Parade':(371.89,200.00),
        'Family Connect Nights':(771.00,1500.00), 'Gingerbread U':(4245.66,4500.00),
        'Homecoming':(266.31,150.00), 'K Playdate':(83.13,300.00),
        'K Orientation':(0.00,400.00), 'Milk & Cookies':(358.07,750.00),
        'Multicultural Night':(2022.51,2500.00), 'Outdoor Movie':(1884.20,2500.00),
        'Science Fair':(883.54,1500.00), 'Spring Fling':(0.00,2500.00),'Spring Dance K-5':(0.00,500.00),
        'Staff Appreciation':(3840.00,4000.00), 'Talent Show':(340.62,1000.00),
        'Talent Show DJ':(550.00,500.00), 'Volunteer Breakfast':(64.00,350.00),
        'Welcome Back Breakfast':(582.66,1000.00), 'WINGO':(0.00,500.00),
    },
    'Donations': {
        'BOE Gifts':(200.00,180.00), 'Cultural Arts':(14651.20,15000.00),
        'Folders':(580.00,600.00), 'Gardening':(0.00,500.00),
        'Hospitality':(124.70,750.00), '5th Staff T-Shirts':(191.78,125.00),
        'Recess Equipment':(1000.00,1000.00), 'School Spirit':(280.76,750.00),
        'Sling Bags':(2354.50,1500.00), 'Spelling Bee':(192.50,225.00),
        'Sunshine Fund':(210.00,500.00), 'Trick or Treat Street':(0.00,250.00),
        'WM Scholarships':(1000.00,1000.00),
    },
    'Grad Class Activities': {
        'Monster Bash':(402.84,1300.00), 'Monster Bash DJ':(500.00,500.00),
        'Electric Parade':(0.00,900.00), 'Winter Social':(1167.86,1300.00),
        'Winter Social DJ':(500.00,500.00), 'Moving Up':(715.00,400.00),
        'Picnic':(2441.98,900.00),
        'Graduating Class Gifts':    (1034.41, 500.00),
        'Graduating Mural':          (244.48,  350.00),
        '5th Grade T-Shirts':        (1494.25, 800.00),
        'Trunk or Treat Fundraiser': (258.67,  500.00),
        'The Night at Rinx':         (710.00,  650.00),
        'Family Contributions':      (0.00,    0.00),   # ← add
        'Lawn Signs':                (0.00,    0.00),   # ← add
        'Treat or Trunk':            (0.00,    0.00),   # ← add
    },
}

def merge_actuals_into_budget(budget_dict, actuals_dict):
    # Apply QB name mapping first
    mapped_actuals = {}
    for qb_name, vals in actuals_dict.items():
        budget_name = QB_TO_BUDGET_MAP.get(qb_name, qb_name)
        if budget_name in mapped_actuals:
            # Sum if multiple QB items map to same budget line
            mapped_actuals[budget_name] = [
                mapped_actuals[budget_name][i] + vals[i] 
                for i in range(12)
            ]
        else:
            mapped_actuals[budget_name] = vals

    result = {}
    all_budget_items = {item for section in budget_dict.values() for item in section}
    for section, items in budget_dict.items():
        result[section] = {}
        for item, (last_yr, budget) in items.items():
            result[section][item] = (last_yr, budget, mapped_actuals.get(item, [0.0]*12))

    unmatched = [k for k in mapped_actuals if k not in all_budget_items]
    if unmatched:
        result['Other (from QuickBooks)'] = {}
        print(f'NOTE: {len(unmatched)} QB item(s) not matched to budget:')
        for item in unmatched:
            print(f'  - {item}')
            result['Other (from QuickBooks)'][item] = (0.0, 0.0, mapped_actuals[item])

    return result

INCOME_MERGED  = merge_actuals_into_budget(INCOME_BUDGET,  INCOME_ACTUALS_LIVE)
EXPENSE_MERGED = merge_actuals_into_budget(EXPENSE_BUDGET, EXPENSE_ACTUALS_LIVE)
print(f'Budget + actuals merged  |  Income sections: {len(INCOME_MERGED)}  |  Expense sections: {len(EXPENSE_MERGED)}')


NOTE: 3 QB item(s) not matched to budget:
  - Folders and planners
  - 5th grade T-shirts
  - BOE
Budget + actuals merged  |  Income sections: 4  |  Expense sections: 6


## Cell 7 — Excel Style Helpers

In [118]:
# Run after Cell 6
for section, items in INCOME_MERGED.items():
    for item in items:
        if any(k in item.lower() for k in ['single', 'family', 'donation', 'teacher', 'student']):
            print(f'  "{item}"  ->  lower: "{item.lower()}"')

  "Single"  ->  lower: "single"
  "Family"  ->  lower: "family"
  "Donations"  ->  lower: "donations"
  "Teachers"  ->  lower: "teachers"
  "Student"  ->  lower: "student"
  "Family Contributions"  ->  lower: "family contributions"


In [119]:
# Run in notebook after Cell 6
print('Family Contributions in INCOME_MERGED:')
for section, items in INCOME_MERGED.items():
    if 'Family Contributions' in items:
        vals = items['Family Contributions']
        print(f'  Section: {section}')
        print(f'  Last yr: {vals[0]}')
        print(f'  Budget:  {vals[1]}')
        print(f'  Actuals: {vals[2]}')
        print(f'  SEPT (index 2): {vals[2][2]}')

print('\nIncome actuals for Family Contributions:')
if 'Family Contributions' in INCOME_ACTUALS_LIVE:
    print(INCOME_ACTUALS_LIVE['Family Contributions'])
else:
    print('NOT FOUND in INCOME_ACTUALS_LIVE')

print('\nQB income categories:')
for k, v in qb['income'].items():
    print(f'  {k}: ${v:,.2f}')

Family Contributions in INCOME_MERGED:
  Section: Grad Class Activities
  Last yr: 6480.0
  Budget:  3900.0
  Actuals: [0.0, 0.0, 3700.0, 300.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  SEPT (index 2): 3700.0

Income actuals for Family Contributions:
NOT FOUND in INCOME_ACTUALS_LIVE

QB income categories:
  Graduating Class Dues: $300.00
  Membership: $45.00
  Family Membership: $50.00
  Standard: $285.00
  Student: $5.00
  Teachers: $240.00
  Book Fair: $28.52
  Birthday Books-Income: $390.00
  Fast Athletics: $910.00
  Trunk or Treat: $430.00


In [120]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

NAVY='1F3864'; TEAL='2E75B6'; LTBLUE='BDD7EE'; GOLD='FFD966'
WHITE='FFFFFF'; LGREY='F2F2F2'; GREEN='E2EFDA'; RED_BG='FCE4D6'

SUBHDR_FONT = Font(name='Arial',bold=True,color=WHITE,size=10)
BODY_FONT   = Font(name='Arial',size=10)
BOLD_FONT   = Font(name='Arial',bold=True,size=10)
TOTAL_FONT  = Font(name='Arial',bold=True,size=10,color=NAVY)

NAVY_FILL   = PatternFill('solid',fgColor=NAVY)
TEAL_FILL   = PatternFill('solid',fgColor=TEAL)
LTBLUE_FILL = PatternFill('solid',fgColor=LTBLUE)
GOLD_FILL   = PatternFill('solid',fgColor=GOLD)
LGREY_FILL  = PatternFill('solid',fgColor=LGREY)
GREEN_FILL  = PatternFill('solid',fgColor=GREEN)
RED_FILL    = PatternFill('solid',fgColor=RED_BG)

THIN        = Side(style='thin',   color='AAAAAA')
MED         = Side(style='medium', color=NAVY)
THIN_BORDER = Border(left=THIN,right=THIN,top=THIN,bottom=THIN)
MED_BORDER  = Border(left=MED, right=MED, top=MED, bottom=MED)
MONEY_FMT   = '$#,##0.00_);($#,##0.00)'

def sec_hdr(ws,label,row,n=4):
    ws.merge_cells(f'A{row}:{get_column_letter(n)}{row}')
    c=ws[f'A{row}']; c.value=label
    c.font=Font(name='Arial',bold=True,size=11,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='left',vertical='center',indent=1)
    ws.row_dimensions[row].height=20; return row+1

def col_hdrs(ws,row,labels):
    for i,lbl in enumerate(labels):
        c=ws.cell(row=row,column=i+1,value=lbl)
        c.font=SUBHDR_FONT; c.fill=TEAL_FILL; c.border=THIN_BORDER
        c.alignment=Alignment(horizontal='center' if i>0 else 'left',vertical='center',indent=1 if i==0 else 0)
    ws.row_dimensions[row].height=18; return row+1

def data_row(ws,row,label,amount,shade=False):
    fill=LGREY_FILL if shade else PatternFill()
    ws[f'A{row}'].value=label; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].fill=fill
    ws[f'A{row}'].border=THIN_BORDER; ws[f'A{row}'].alignment=Alignment(indent=2)
    ws[f'B{row}'].value=amount; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].fill=fill
    ws[f'B{row}'].border=THIN_BORDER; ws[f'B{row}'].number_format=MONEY_FMT
    ws[f'B{row}'].alignment=Alignment(horizontal='right')
    for col in ['C','D']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
    ws.row_dimensions[row].height=16

def total_row(ws,row,label,val):
    for col in ['A','B','C','D']:
        ws[f'{col}{row}'].fill=LTBLUE_FILL; ws[f'{col}{row}'].border=MED_BORDER
    ws[f'A{row}'].value=label; ws[f'A{row}'].font=TOTAL_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
    ws[f'B{row}'].value=val; ws[f'B{row}'].font=TOTAL_FONT
    ws[f'B{row}'].number_format=MONEY_FMT; ws[f'B{row}'].alignment=Alignment(horizontal='right')
    ws.row_dimensions[row].height=18

print('Styles ready')


Styles ready


## Cell 8 — Sheet Builders

In [121]:
print('Sheet builders ready')


Sheet builders ready


## Cell 9 — Generate Excel Report

In [122]:
print(f'Building workbook  ->  {ORG_NAME}  |  {MONTH_LABEL}  |  Column: {FISCAL_MONTHS[FISCAL_IDX]}')
wb = openpyxl.Workbook()

ws1 = wb.active; ws1.title = 'Treasurer Report'
build_treasurer(ws1, qb, bank, MONTH_LABEL, ORG_NAME)
print('  Tab 1: Treasurer Report')

ws2 = wb.create_sheet('Income Budget vs Actuals')
build_budget(ws2, 'Budget vs Actuals - Income', INCOME_MERGED, ORG_NAME, FISCAL_MONTHS, FISCAL_IDX, show_pl=False)

ws3 = wb.create_sheet('Expense Budget vs Actuals')
build_budget(ws3, 'Budget vs Actuals - Expenses', EXPENSE_MERGED, ORG_NAME, FISCAL_MONTHS, FISCAL_IDX, show_pl=True, income_merged=INCOME_MERGED)


ws4 = wb.create_sheet('Giveback Reconciliation')
build_givebacks(ws4, givebacks, bank, ORG_NAME)
print('  Tab 4: Giveback Reconciliation')

ws5 = wb.create_sheet('File Manifest')
build_manifest(ws5, MONTH_FOLDER, ORG_NAME, MONTH_LABEL, FISCAL_IDX, FISCAL_MONTHS)
print('  Tab 5: File Manifest')

ws6 = wb.create_sheet('YTD Summary')
build_ytd_summary(ws6, INCOME_MERGED, EXPENSE_MERGED, ORG_NAME,
                  MONTH_LABEL, FISCAL_IDX, FISCAL_MONTHS,
                  bank=bank,
                  balance_forward=32630.10)  # ← update this each fiscal year
print('  Tab 6: YTD Summary')

wb.save(OUTPUT_FILE)
print(f'\nSaved -> {OUTPUT_FILE}')


Building workbook  ->  Setauket School PTA  |  October 2025  |  Column: OCT
  Tab 1: Treasurer Report
  Tab 4: Giveback Reconciliation
  Tab 5: File Manifest
  Tab 6: YTD Summary

Saved -> output/Treasurer_Report_October_2025.xlsx


In [123]:
# Run in a new cell
print('Income categories from QB:')
for k, v in qb['income'].items():
    print(f'  {k:<40} ${v:,.2f}')

print('\nAll transactions with Membership:')
for t in qb['transactions']:
    if 'member' in t['category'].lower() or 'member' in t['description'].lower():
        print(f"  {t['date']}  {t['category']:<30}  {t['payee']:<25}  ${t['amount']:,.2f}")

Income categories from QB:
  Graduating Class Dues                    $300.00
  Membership                               $45.00
  Family Membership                        $50.00
  Standard                                 $285.00
  Student                                  $5.00
  Teachers                                 $240.00
  Book Fair                                $28.52
  Birthday Books-Income                    $390.00
  Fast Athletics                           $910.00
  Trunk or Treat                           $430.00

All transactions with Membership:
  10/03/2025  Graduating Class Dues           MemberHub                  $100.00
  10/10/2025  Graduating Class Dues           MemberHub                  $150.00
  10/31/2025  Graduating Class Dues           MemberHub                  $50.00
  10/24/2025  Membership                      MemberHub                  $45.00
  10/10/2025  Family Membership               MemberHub                  $50.00
  10/14/2025  Standard         

In [124]:
print('Unmatched items (in Other):')
for section, items in INCOME_MERGED.items():
    if 'Other' in section:
        for item in items:
            print(f'  INCOME: {item}')
for section, items in EXPENSE_MERGED.items():
    if 'Other' in section:
        for item in items:
            print(f'  EXPENSE: {item}')

Unmatched items (in Other):
  EXPENSE: Folders and planners
  EXPENSE: 5th grade T-shirts
  EXPENSE: BOE


In [125]:
print('Income merged keys:')
for section, items in INCOME_MERGED.items():
    for item in items:
        print(f'  {item}')

print('\nExpense merged keys:')
for section, items in EXPENSE_MERGED.items():
    for item in items:
        print(f'  {item}')

Income merged keys:
  Birthday Books
  Book Fair
  Croc Charms
  Fall Pictures
  FAST
  Holiday Boutique
  Plant Sale
  Spiritwear
  Spring Pictures
  Ticket & Raffle Sales
  Sponsors
  Single
  Family
  Donations
  Teachers
  Student
  Gingerbread U
  Staff Appreciation
  Talent Show
  Family Contributions
  Lawn Signs
  Treat or Trunk
  The Night at Rinx

Expense merged keys:
  Accountant
  Bank Services
  Insurance
  Supplies
  Accounting Quickbooks
  Training
  Website & Remind App
  Event Equipment
  Council Dues
  Membership Expenses
  Entertainment
  Raffles
  Venue
  Birthday Books
  Book Fair
  Croc Charms
  Fall Pictures
  FAST
  Holiday Boutique
  Plant Sale
  Spiritwear
  Spring Pictures
  Bus Driver Appreciation
  Electric Parade
  Family Connect Nights
  Gingerbread U
  Homecoming
  K Playdate
  K Orientation
  Milk & Cookies
  Multicultural Night
  Outdoor Movie
  Science Fair
  Spring Fling
  Spring Dance K-5
  Staff Appreciation
  Talent Show
  Talent Show DJ
  Volunte

## Cell 10 — Financial Summary

In [126]:
print('='*55)
print(f'  {ORG_NAME}  -  {MONTH_LABEL}')
print(f'  Budget column updated: {FISCAL_MONTHS[FISCAL_IDX]} (index {FISCAL_IDX})')
print('='*55)
print(f'  Income            : ${qb["income_total"]:>12,.2f}')
print(f'  Expenses          : ${qb["expense_total"]:>12,.2f}')
print(f'  Net Income (Loss) : ${qb["net_income"]:>12,.2f}')
print('-'*55)
print(f'  Bank Beginning    : ${bank["beginning_balance"]:>12,.2f}')
print(f'  Bank Ending       : ${bank["ending_balance"]:>12,.2f}')
calc = bank['beginning_balance']+bank['total_deposits']-bank['total_checks']-bank.get('total_withdrawals', 0.0)-bank['total_fees']
diff = calc - bank['ending_balance']
print(f'  Reconciliation    : {"Balanced" if abs(diff)<0.01 else f"Off by ${abs(diff):,.2f}"}')
print(f'  Givebacks Total   : ${sum(g["total"] for g in givebacks):>12,.2f}')
print('='*55)
hist_files = sorted(HISTORY_DIR.glob('*.json'))
print(f'\n  Months stored in history: {len(hist_files)}')
for hf in hist_files:
    e = json.loads(hf.read_text())
    fi = e.get("fiscal_index", "?")
    col = FISCAL_MONTHS[fi] if isinstance(fi, int) else "?"
    print(f'    [{col:<5}] {e["month_label"]:<20} income=${e["income_total"]:>10,.2f}  expenses=${e["expense_total"]:>10,.2f}')
print(f'\n  Output: {OUTPUT_FILE}')


  Setauket School PTA  -  October 2025
  Budget column updated: OCT (index 3)
  Income            : $    2,683.52
  Expenses          : $   17,851.64
  Net Income (Loss) : $  -15,168.12
-------------------------------------------------------
  Bank Beginning    : $   49,954.22
  Bank Ending       : $   34,446.10
  Reconciliation    : Balanced
  Givebacks Total   : $    1,250.00

  Months stored in history: 6
    [APR  ] April 2026           income=$ 31,732.00  expenses=$ 21,534.79
    [AUG  ] August 2025          income=$    525.14  expenses=$    391.37
    [JULY ] July 2025            income=$      0.00  expenses=$  1,439.34
    [MAR  ] March 2026           income=$  8,405.21  expenses=$  2,552.28
    [OCT  ] October 2025         income=$  2,683.52  expenses=$ 17,851.64
    [SEPT ] September 2025       income=$ 20,661.97  expenses=$  2,032.28

  Output: output/Treasurer_Report_October_2025.xlsx


In [127]:
import os
hist_files = sorted(Path('data/history').glob('*.json'))
for hf in hist_files:
    print(f'{hf.name}  ({os.path.getsize(hf)} bytes)')
print('Expenses from QB detail:')
for k, v in qb['expenses'].items():
    print(f'  {k:<35} ${v:,.2f}')
print(f'\nTotal: ${qb["expense_total"]:,.2f}')    

April_2026.json  (935 bytes)
August_2025.json  (396 bytes)
July_2025.json  (311 bytes)
March_2026.json  (510 bytes)
October_2025.json  (925 bytes)
September_2025.json  (680 bytes)
Expenses from QB detail:
  Movie Night                         $414.29
  Scholastic Book Fairs               $3,885.07
  MEMBERSHIP EXPENSE                  $327.00
  Fast Athletics                      $9,375.00
  Folders and planners                $640.00
  Homecoming                          $67.20
  Spelling Bee                        $206.50
  Trunk or Treat                      $258.67
  Welcome Back Staff                  $823.40
  5th grade T-shirts                  $901.00
  Monster Bash                        $273.51
  Monster Bash DJ                     $500.00
  BOE                                 $180.00

Total: $17,851.64


## Cell 11 — Push to GitHub
Pushes code changes to your GitHub repository via SSH.

**What gets pushed:** notebook, README, .gitignore, requirements.txt — never input files, output Excel, or `.env`.

**One-time setup (run in terminal, not here):**
```bash
git init
git remote add origin git@github.com:yourname/pta-treasurer.git
# Add your SSH public key at: github.com → Settings → SSH and GPG keys
```


In [128]:
import subprocess

# Files/folders that should NEVER be pushed
SENSITIVE_PATTERNS = ['input/', 'output/', 'data/', '.env', '*.xlsx', '*.pdf', '*.csv','debug_']

def run_git(args, check=True):
    """Run a git command and return (returncode, stdout, stderr)."""
    result = subprocess.run(
        ['git'] + args,
        capture_output=True, text=True
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'git {" ".join(args)} failed:\n{result.stderr}')
    return result.returncode, result.stdout.strip(), result.stderr.strip()

def push_to_github(commit_message=None):
    print('GitHub Push')
    print('='*55)

    # 1. Check git is initialized
    code, _, _ = run_git(['rev-parse', '--git-dir'], check=False)
    if code != 0:
        print('ERROR: Not a git repository.')
        print('Run in terminal:')
        print('  git init')
        print(f'  git remote add {GITHUB_REMOTE} git@github.com:yourname/pta-treasurer.git')
        return

    # 2. Check remote exists
    _, remotes, _ = run_git(['remote'])
    if GITHUB_REMOTE not in remotes.split():
        print(f'ERROR: Remote "{GITHUB_REMOTE}" not found.')
        print(f'Run in terminal: git remote add {GITHUB_REMOTE} git@github.com:yourname/pta-treasurer.git')
        return

    # 3. Check .gitignore exists and protects sensitive files
    gitignore = Path('.gitignore')
    if not gitignore.exists():
        print('Creating .gitignore...')
        gitignore.write_text(
            '# Input files - never commit\n'
            'input/\n'
            'output/\n'
            'data/\n'
            '*.xlsx\n'
            '*.pdf\n'
            '*.csv\n'
            '.env\n'
            '__pycache__/\n'
            '*.pyc\n'
            '.DS_Store\n'
        )
        print('  .gitignore created')

    # 4. Show git status
    _, status, _ = run_git(['status', '--short'])
    if not status:
        print('Nothing to commit - working tree clean.')
        return

    print('\nFiles changed:')
    for line in status.split('\n'):
        if not line.strip(): continue
        parts = line.split(None, 1)
        if len(parts) < 2: continue
        flag  = parts[0].strip()
        fname = parts[1].strip()
        is_sensitive = any(
            fname.startswith(p.rstrip('*').rstrip('/')) or
            fname.endswith(p.lstrip('*'))
            for p in SENSITIVE_PATTERNS
        )
        icon = '  SKIP (sensitive)' if is_sensitive else '  will push'
        print(f'  {flag} {fname:<45} {icon}')

    # 5. Ask for commit message
    if commit_message is None:
        default_msg = f'Update report code - {MONTH_LABEL}'
        commit_message = input(f'\nCommit message [{default_msg}]: ').strip()
        if not commit_message:
            commit_message = default_msg

    # 6. Stage only safe files (not sensitive paths)
    print('\nStaging files...')
    safe_to_add = []
    for line in status.split('\n'):
        if not line.strip(): continue
        parts = line.split(None, 1)
        if len(parts) < 2: continue
        flag  = parts[0].strip()
        fname = parts[1].strip()
        if not fname: continue
        # Only stage modified and added files — NOT untracked (??)
        if flag not in ('M', 'A', 'AM', 'MM', 'R'): continue
        is_sensitive = any(
            fname.startswith(p.rstrip('*').rstrip('/')) or
            fname.endswith(p.lstrip('*'))
            for p in SENSITIVE_PATTERNS
        )
        if not is_sensitive:
            safe_to_add.append(fname)
        
    if not safe_to_add:
        print('No safe files to push (all changes are in sensitive paths).')
        return

    for f in safe_to_add:
        run_git(['add', f])
        print(f'  staged: {f}')

    # 7. Commit
    print(f'\nCommitting: "{commit_message}"')
    run_git(['commit', '-m', commit_message])

    # 8. Push
    print(f'Pushing to {GITHUB_REMOTE}/{GITHUB_BRANCH}...')
    run_git(['push', GITHUB_REMOTE, GITHUB_BRANCH])

    # 9. Show result
    _, log, _ = run_git(['log', '--oneline', '-1'])
    _, remote_url, _ = run_git(['remote', 'get-url', GITHUB_REMOTE])
    repo_url = remote_url.replace('git@github.com:', 'https://github.com/').replace('.git','')

    print('='*55)
    print(f'  Pushed successfully!')
    print(f'  Commit : {log}')
    print(f'  Repo   : {repo_url}')
    print('='*55)


# ── Run ───────────────────────────────────────────────────────────────────────
#push_to_github()
if not os.getenv('BATCH_MODE'):
    push_to_github()

GitHub Push

Files changed:
  M PTA_Treasurer_Report_v4.ipynb                   will push
  M parsers.py                                      will push
  ?? logs/                                           will push
  ?? run_all_months.sh                               will push

Commit message [Update report code - October 2025]: Included changes to automate running over all. months

Staging files...
  staged: PTA_Treasurer_Report_v4.ipynb
  staged: parsers.py

Committing: "Included changes to automate running over all. months"
Pushing to origin/main...
  Pushed successfully!
  Commit : 3aacce7 Included changes to automate running over all. months
  Repo   : https://github.com/deepssharma/pta_treasurer


In [ ]:
import subprocess
result = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if line:
        print(repr(line))  # repr shows exact characters including spaces

In [ ]:
_, status, _ = run_git(['status', '--short'])
for line in status.split('\n'):
    if not line.strip(): continue
    print(repr(line))
    print(f'  line[0]={repr(line[0])} line[1]={repr(line[1])} line[2]={repr(line[2])} line[3:]={repr(line[3:])}')